In [51]:
from langchain_community.tools import WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper

In [52]:
apiwrapper= WikipediaAPIWrapper(top_k_results=1, doc_content_chars_max=300)
wiki_tool=WikipediaQueryRun(api_wrapper=apiwrapper)

In [53]:
#using external documents(custom tool)
from langchain_community.document_loaders import WebBaseLoader
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import OllamaEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter


loader=WebBaseLoader("https://docs.langchain.com/oss/python/langchain/overview")
docs=loader.load()
documents=RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200).split_documents(docs)
embeddings_model=OllamaEmbeddings(model='nomic-embed-text')
vectordb=FAISS.from_documents(documents, embeddings_model)
retriever=vectordb.as_retriever()
retriever

VectorStoreRetriever(tags=['FAISS', 'OllamaEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x75354d5daf90>, search_kwargs={})

In [54]:
from langchain_core.tools.retriever import create_retriever_tool

retrieval_tool=create_retriever_tool(retriever, "langsmith_search", 
"search for information about LangSmith. for any question regarding LangSmith use this tool")

In [55]:
from langchain_community.tools import ArxivQueryRun
from langchain_community.utilities import ArxivAPIWrapper

arxiv_wrapper=ArxivAPIWrapper(top_k_results=1, doc_content_chars_max=400)
arxiv_tool=ArxivQueryRun(api_wrapper=arxiv_wrapper)

In [56]:
tools=[wiki_tool, retrieval_tool, arxiv_tool]
tools

[WikipediaQueryRun(api_wrapper=WikipediaAPIWrapper(wiki_client=<module 'wikipedia' from '/home/ojass-bhardwaj/Desktop/langchain/denv/lib/python3.12/site-packages/wikipedia/__init__.py'>, top_k_results=1, lang='en', load_all_available_meta=False, doc_content_chars_max=300)),
 StructuredTool(name='langsmith_search', description='search for information about LangSmith. for any question regarding LangSmith use this tool', args_schema=<class 'langchain_core.tools.retriever.RetrieverInput'>, func=<function create_retriever_tool.<locals>.func at 0x7534e75ed580>, coroutine=<function create_retriever_tool.<locals>.afunc at 0x7534e75edc60>),
 ArxivQueryRun(api_wrapper=ArxivAPIWrapper(arxiv_search=<class 'arxiv.Search'>, arxiv_exceptions=(<class 'arxiv.ArxivError'>, <class 'arxiv.UnexpectedEmptyPageError'>, <class 'arxiv.HTTPError'>), top_k_results=1, ARXIV_MAX_QUERY_LENGTH=300, continue_on_failure=False, load_max_docs=100, load_all_available_meta=False, doc_content_chars_max=400))]

In [57]:
from dotenv import load_dotenv
load_dotenv()
import os
os.environ['GROQ_API_KEY']=os.getenv("GROQ_API_KEY")
os.environ['LANGCHAIN_API_KEY']=os.getenv("LANGCHAIN_API_KEY")
os.environ['LANGCHAIN_TRACING_V2']="True"
from langchain_groq import ChatGroq

llm = ChatGroq(model='llama-3.3-70b-versatile')


In [58]:
import os
from dotenv import load_dotenv, find_dotenv
from langsmith import Client

# 1. Clear any active cache of the broken endpoint
if "LANGCHAIN_ENDPOINT" in os.environ:
    del os.environ["LANGCHAIN_ENDPOINT"]

# 2. Safely read keys from your .env file
load_dotenv(find_dotenv())

# 3. Pull the prompt via the standard client
client = Client()
prompt = client.pull_prompt("rlm/rag-prompt", dangerously_pull_public_prompt=True)

# 4. Access the message structures safely
prompt.messages

[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, template="You are an assistant for question-answering tasks. Use the following pieces of retrieved context to answer the question. If you don't know the answer, just say that you don't know. Use three sentences maximum and keep the answer concise.\nQuestion: {question} \nContext: {context} \nAnswer:"), additional_kwargs={})]

In [49]:
from langchain_classic.agents import create_tool_calling_agent, AgentExecutor
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        "You are a helpful and precise assistant. You have access to a set of tools.\n\n"
        "CRITICAL RULES:\n"
        "1. If you need to use a tool, output the tool call IMMEDIATELY.\n"
        "2. Do NOT explain your reasoning, do NOT write any introductory text, "
        "and do NOT say 'I will now call the tool...'. Just execute the tool call.\n"
        "3. Only provide conversational explanations AFTER you have received the tool's output."
    ),
    # Optional: For keeping track of previous turns
    MessagesPlaceholder(variable_name="chat_history", optional=True),
    
    # The user's current question
    ("human", "{input}"),
    
    # CRITICAL: Must be at the very end to store intermediate tool steps
    MessagesPlaceholder(variable_name="agent_scratchpad")
])

agent=create_tool_calling_agent(llm, tools, prompt)
agent_executor=AgentExecutor(agent=agent, tools=tools, verbose=True)
agent_executor



AgentExecutor(verbose=True, agent=RunnableMultiActionAgent(runnable=RunnableAssign(mapper={
  agent_scratchpad: RunnableLambda(lambda x: message_formatter(x['intermediate_steps']))
})
| ChatPromptTemplate(input_variables=['agent_scratchpad', 'input'], optional_variables=['chat_history'], input_types={'chat_history': list[typing.Annotated[typing.Union[typing.Annotated[langchain_core.messages.ai.AIMessage, Tag(tag='ai')], typing.Annotated[langchain_core.messages.human.HumanMessage, Tag(tag='human')], typing.Annotated[langchain_core.messages.chat.ChatMessage, Tag(tag='chat')], typing.Annotated[langchain_core.messages.system.SystemMessage, Tag(tag='system')], typing.Annotated[langchain_core.messages.function.FunctionMessage, Tag(tag='function')], typing.Annotated[langchain_core.messages.tool.ToolMessage, Tag(tag='tool')], typing.Annotated[langchain_core.messages.ai.AIMessageChunk, Tag(tag='AIMessageChunk')], typing.Annotated[langchain_core.messages.human.HumanMessageChunk, Tag(tag='HumanMe

In [61]:
agent_executor.invoke({"input":"Tell me about Langsmith"})



> Entering new AgentExecutor chain...

Invoking: `langsmith_search` with `{'query': 'Langsmith information'}`


See the Installation instructions and Quickstart guide to get started building your own agents and applications with LangChain.
Use LangSmith to trace requests, debug agent behavior, and evaluate outputs. Set LANGSMITH_TRACING=true and your API key to get started.
​ Core benefits

Use LangSmith to trace requests, debug agent behavior, and evaluate outputs. Set LANGSMITH_TRACING=true and your API key to get started.
​ Core benefits
Standard model interfaceUse one interface for chat models, embeddings, and more across providers. Switch models with minimal code changes and keep your application portable as requirements evolve.Learn moreHighly configurable harnessStart with create_agent as a minimal harness and add capabilities incrementally through middleware. Compose only what your use case needs, from guardrails and retries to routing and custom tool policies.Learn moreBuilt

{'input': 'Tell me about Langsmith',
 'output': 'LangSmith is a tool used to trace requests, debug agent behavior, and evaluate outputs in LangChain, a framework for building agents and applications. It provides a standard model interface, a highly configurable harness, and is built on top of LangGraph, which allows for durable execution, human-in-the-loop support, persistence, and more. LangSmith can be used to inspect traces, tool calls, state transitions, and latency in one place, helping to find failure modes, evaluate quality, and improve agent behavior. It is recommended to set LANGSMITH_TRACING=true and provide an API key to get started with LangSmith.'}